In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@monarchazuredatalake.dfs.core.windows.net/customers")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")

###### I want to extract the domain names from the mail so that i can give different coupons based on each domain name

Lets use split() --> array Indexing, 

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("domain", split(col("email"), "@")[1])

##### Now grouping by the domain and sort, and apply aggregation

In [0]:
df.groupBy("domain").agg(count("customer_id").alias("total_customers")).sort("total_customers", ascending=False).display()

###### Now applying the FILTER(COL() ...) to get the specific data 

In [0]:
df.filter(col("domain") == "gmail.com").display()

###### Concatinate

In [0]:
df = df.withColumn("full_name",concat (col("first_name"), lit(" "), col("last_name")))
df = df.drop("first_name").drop("last_name")
df.display()


##### Finally writing into the datalake

In [0]:
df.write.mode("append").format("delta").save("abfss://silver@monarchazuredatalake.dfs.core.windows.net/customers")